# KKT stationarity of `P_g`

Decomposes `∂L/∂P_g` into its dual terms for one TSC-OPF run and checks that the
parts add up.

Everything the notebook needs about the model that produced the run is read from
`run_manifest.toml` — nothing below is hardcoded except `RUN_DIR`, so the same
notebook works on Kron, FULL_BUS and DQ_4TH runs, with or without AVR, governor
and grid-forming converters.

Requires: `numpy`, `pandas`, `matplotlib`. TOML is read with the standard-library
`tomllib` (Python 3.11+). The XLSX mirrors are never touched.

In [ ]:
import os
from pathlib import Path
import tomllib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Where to look. Precedence:
#   1. the KKT_RUN_DIR environment variable, if set — this is how runs produced by
#      another project
#      are analysed without editing the notebook;
#   2. otherwise this repository's own RESULTS.
# Either may name a single `Results - *` folder or a RESULTS root; a root is
# resolved to the most recent run that carries a KKT export, which is not the same
# as the most recent run — a test sweep or a plain main.jl run is often newer and
# has no kkt_Pg_terms.csv in it.
RUN_DIR = Path(os.environ.get("KKT_RUN_DIR", r"..\RESULTS"))

if RUN_DIR.is_dir() and not (RUN_DIR / "run_manifest.toml").exists():
    candidates = sorted(p.parents[2] for p in
                        RUN_DIR.glob("Results - */Dispatch/CSV_duals/kkt_Pg_terms.csv"))
    if not candidates:
        raise FileNotFoundError(
            f"No run under {RUN_DIR.resolve()} contains "
            "Dispatch/CSV_duals/kkt_Pg_terms.csv. Produce one with "
            "`julia --project=TSCOPF runs/run_9bus.jl linear`, or attach the hook to your "
            "own run script, or point KKT_RUN_DIR / RUN_DIR somewhere else."
        )
    RUN_DIR = candidates[-1]

manifest_path = RUN_DIR / "run_manifest.toml"
if not manifest_path.exists():
    raise FileNotFoundError(
        f"{manifest_path} is missing. run_manifest.toml is written by "
        "Save_Run_Manifest! on every run; a folder without one predates that "
        "export and cannot be read back reliably (load_factor is not recorded "
        "anywhere else). Re-run the case with a current build."
    )

with open(manifest_path, "rb") as fh:
    M = tomllib.load(fh)

run = M["run"]
res = M["resolved"]
dyn = M.get("transient", {}).get("dyn_model", {})
fault = dyn.get("fault", {})

# Manifest keys are ASCII: the writer transliterates the Greek field names of the
# model (bound_style_δ -> bound_style_delta, Δω_tol -> Delta_omega_tol). The banner
# is ASCII too, so it survives a cp1252 console as well as a notebook.
STYLE_DELTA = dyn.get("bound_style_delta", "-") if dyn.get("constrain_delta") else "off"
STYLE_OMEGA = (dyn.get("bound_style_Delta_omega", "-")
               if dyn.get("constrain_Delta_omega") else "off")

print(f"run dir        {RUN_DIR}")
print(f"case           {run['case']} x load_factor {run['load_factor']}")
print(f"dispatch       {M['dispatch']['type_model']} ({M['dispatch']['cost_type']} cost)")
print(f"machine model  {dyn.get('gen_order', '-')} on {dyn.get('network_form', '-')}")
print(f"controls       AVR={dyn.get('include_avr', False)}  "
      f"governor={dyn.get('include_governor', False)}")
print(f"corridors      delta:{STYLE_DELTA}  domega:{STYLE_OMEGA}")
print(f"fleet          SG {res.get('sg_ids', [])}  GFM {res.get('gfm_ids', [])}")
print(f"disturbance    {fault.get('fault_type', '-')}")
print(f"timeline       dt={res.get('t_step')} s, "
      f"{res.get('n_steps_fault', 0)} fault + "
      f"{res.get('n_steps_postfault', 0)} post-fault steps")
print(f"status         {M['status'].get('termination_status')}  "
      f"obj {M['status'].get('objective_MVA')}")

In [ ]:
def rd(path):
    """Every CSV this codebase writes is ';'-delimited."""
    return pd.read_csv(path, sep=";")


TS_CSV = RUN_DIR / "Transient_Stability" / "CSV"
TS_DUALS = RUN_DIR / "Transient_Stability" / "CSV_duals"
DISPATCH_DUALS = RUN_DIR / "Dispatch" / "CSV_duals"

kkt = rd(DISPATCH_DUALS / "kkt_Pg_terms.csv")

# --- residual check ----------------------------------------------------------
# contribution(family) = -dual x d(expr)/dP_g, and the sum plus dC/dP_g must vanish.
residual = kkt[kkt.family == "RESIDUAL"].set_index("gen_id").contribution
worst = residual.abs().max()
assert worst < 1e-6, f"stationarity residual {worst:.3e} exceeds 1e-6:\n{residual}"
print(f"residual check passed — worst |∂L/∂P_g| = {worst:.3e} €/MW\n")

# --- the decomposition table -------------------------------------------------
terms = kkt[~kkt.family.isin(["RESIDUAL"])]
table = terms.pivot_table(index="gen_id", columns="family",
                          values="contribution", aggfunc="sum").fillna(0.0)

# Four buckets, not three. `Pm_init` (and its converter twin `gfm_Pmeas_init`) is
# NOT dynamic coupling: it is the equality P_m = P_g, so its dual ν is whatever
# stationarity in P_m makes it — and with `bound_P_m` on and no governor that is
# exactly the rent of the box sitting on P_m. It therefore carries a *capacity*
# price into the P_g row with no transient content at all. Lumping it under
# "dynamic" reads a stability rent into runs where the corridor never binds.
SCARCITY = ["pg_lower", "pg_upper", "sg_upper", "gfm_Imax", "gfm_Emax"]
MECH_PIN = ["Pm_init", "gfm_Pmeas_init"]
DYNAMIC = ["Pe_init", "swing_first_step", "swing_tf", "swing_tpf",
           "corridor_tf_lower", "corridor_tf_upper",
                "corridor_tpf_lower", "corridor_tpf_upper"]

present = lambda names: [c for c in names if c in table.columns]

split = pd.DataFrame(index=table.index)
split["cost"] = table.get("cost_gradient", 0.0)
split["scarcity"] = table[present(SCARCITY)].sum(axis=1)
split["mech_pin"] = table[present(MECH_PIN)].sum(axis=1)
split["dynamic"] = table[present(DYNAMIC)].sum(axis=1)
# p_balance contributes -λ, so the nodal price is the negated contribution.
split["lambda"] = -table["p_balance"]
split["check"] = (split["cost"] + split["scarcity"] + split["mech_pin"]
                  + split["dynamic"] - split["lambda"])

print("per-family contributions to ∂L/∂P_g  [€/MW]")
display(table.round(4))
print("\nλ split  [€/MW]   (check ≈ 0 by construction)")
display(split.round(4))

# --- is the mechanical pin just re-charging the P_g capacity limit? -----------
# `bound_P_g` (DispatchConfig) and `bound_P_m` (TsBuilderConfig, with
# P_m_source = :dgen_pg_limits) impose the SAME physical Pmax on two variables
# tied by P_m = P_g. The feasible set is unchanged, but the multiplier is
# degenerate: any split of the capacity rent between η⁺ and ν satisfies
# stationarity, and an interior-point solver typically splits it evenly — which
# is why the two columns can come out identical. Only the SUM is meaningful.
pin = split["mech_pin"]
if pin.abs().max() > 1e-6:
    # Stationarity in P_m, with contribution = -dual x d(expr)/dP_m:
    #     -ν - η_u + η_l = 0   =>   ν = -η_u + η_l
    # The two sides enter with opposite signs because the upper row is
    # (P_m - max) ≤ 0 while the lower row is (min - P_m) ≤ 0.
    pm_rent = pd.Series(0.0, index=split.index)
    for name, sign in (("dual_UB_Pm.csv", -1.0), ("dual_LB_Pm.csv", +1.0)):
        path = TS_DUALS / name
        if path.exists():
            d = rd(path)
            if "Gen_ID" in d.columns:
                pm_rent = pm_rent.add(
                    sign * d.set_index("Gen_ID").iloc[:, 0], fill_value=0.0)
    gap = (pin - pm_rent).abs().max() if len(pm_rent) else float("nan")
    print(f"\nmechanical-pin diagnostic: max |ν - (P_m box rent)| = {gap:.3e}")
    if gap < 1e-6:
        print("  The pin is transmitting the P_m box, not transient physics. The")
        print("  unit's capacity rent is split arbitrarily between 'scarcity' and")
        print("  'mech_pin' — read the two as one number:")
    else:
        print("  P_m carries more than its own box here (a governor also ties")
        print("  P_ref and the valve to it), so the pin is not purely a capacity")
        print("  rent. Treat scarcity + mech_pin as the combined non-cost margin:")
    print("  " + str((split["scarcity"] + split["mech_pin"]).round(4).to_dict()))

if split["dynamic"].abs().max() < 1e-6:
    print("\nNOTE: the dynamic terms are numerically zero — the transient")
    print("constraints carry no price in this run. Confirm on the corridor CSVs")
    print("before reading any stability rent into λ.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# (a) every family, stacked, one bar per generator.
families = [c for c in table.columns if c != "cost_gradient"] + ["cost_gradient"]
bottom_pos = np.zeros(len(table))
bottom_neg = np.zeros(len(table))
x = np.arange(len(table))
for name in families:
    values = table[name].to_numpy()
    base = bottom_pos.copy()
    base[values < 0] = bottom_neg[values < 0]
    axes[0].bar(x, values, bottom=base, label=name)
    bottom_pos = np.where(values > 0, bottom_pos + values, bottom_pos)
    bottom_neg = np.where(values < 0, bottom_neg + values, bottom_neg)
axes[0].set_xticks(x, [f"G{g}" for g in table.index])
axes[0].axhline(0.0, color="k", lw=0.8)
axes[0].set_ylabel("contribution to ∂L/∂P_g  [€/MW]")
axes[0].set_title("Stationarity terms (sum = 0)")
axes[0].legend(fontsize=7, ncol=2)

# (b) the same information as a price decomposition. scarcity and mech_pin are
# adjacent on purpose: when both P_g and P_m carry the DGEN capacity box, the
# rent is split arbitrarily between them and only their sum is meaningful.
split[["cost", "scarcity", "mech_pin", "dynamic"]].plot(kind="bar", stacked=True,
                                                        ax=axes[1])
axes[1].plot(x, split["lambda"].to_numpy(), "k_", ms=22, label="λ")
axes[1].set_xticklabels([f"G{g}" for g in split.index], rotation=0)
axes[1].set_ylabel("€/MW")
axes[1].set_title("λ = cost + capacity rent (scarcity + mech_pin) + dynamic")
axes[1].legend(fontsize=8)

fig.tight_layout()

In [ ]:
# (c) the corridor duals this run actually built, over time.
#
# Which delta / domega families exist depends on the corridor style, so the file
# list comes from [exports] rather than from a guess.
written = set(M["exports"].get("ts_dual_files", []))

wanted = []
if STYLE_DELTA in ("coi_box", "swing_propagated"):
    wanted += ["dual_delta_COI_lower.csv", "dual_delta_COI_upper.csv"]
elif STYLE_DELTA in ("highest_H", "ref_gen"):
    wanted += ["dual_delta_ref_lower.csv", "dual_delta_ref_upper.csv"]
if STYLE_OMEGA == "coi_box":
    wanted += ["dual_Delta_Omega_COI_lower.csv", "dual_Delta_Omega_COI_upper.csv"]
elif STYLE_OMEGA == "abs":
    wanted += ["dual_Delta_Omega_abs_lower.csv", "dual_Delta_Omega_abs_upper.csv"]
wanted = [f for f in wanted if f in written]

# The dual rows are row-aligned with the trajectory CSVs; t comes from there.
t = rd(TS_CSV / "electrical_power.csv")["t"].to_numpy()

if not wanted:
    print("no corridor dual files in this run")
else:
    fig, axes = plt.subplots(len(wanted), 1, figsize=(11, 2.6 * len(wanted)),
                             sharex=True, squeeze=False)
    for ax, name in zip(axes[:, 0], wanted):
        df = rd(TS_DUALS / name)
        n = min(len(t), len(df))
        for column in df.columns:
            values = df[column].to_numpy()[:n]
            if np.abs(values).max() <= 0.0:
                continue  # never binding: leave it out of the legend
            ax.plot(t[:n], values, lw=1.2, label=column)
        ax.set_title(name, fontsize=9)
        ax.set_ylabel("dual")
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize=7, ncol=4)
    axes[-1, 0].set_xlabel("t [s]")
    fig.tight_layout()

## Reading the split — two traps

### 1. The mechanical pin is not dynamics

`Pm_init` is the equality `P_m = P_g`. Stationarity in `P_m` fixes its dual as
`ν = −η_m`, the rent of whatever box sits on `P_m`. So when `bound_P_m` is on with
`P_m_source = :dgen_pg_limits`, the *same* physical `Pmax` constrains `P_g` and
`P_m`, and the unit's capacity rent enters the `P_g` row **twice** — once as `η⁺`
and once through the pin.

The feasible set is unchanged and λ is correct, but the multiplier is degenerate:
any split of the rent between `η⁺` and `ν` satisfies stationarity, and an
interior-point solver typically splits it evenly. **Only `scarcity + mech_pin`
is economically meaningful**; neither half alone is. If a clean attribution
matters for a case study, drop one of the two boxes (`bound_P_m = false`, or
widen `P_m` beyond the dispatch limits) and re-solve.

### 2. Γ never appears in the `P_g` row here

In this formulation δ and Δω are **explicit variables**, each with its own defining
equality, so the stability-corridor duals Γ (`dual_delta_COI_*`, `dual_delta_ref_*`,
`dual_Delta_Omega_*`) do not appear in the stationarity condition for `P_g`
directly. That differs from the substituted formulation of the earlier linear-test
notebook, where the corridor was written against a closed-form `h_P` and therefore
injected `c₁Γ` and `c₃Γ` straight into it.

Here Γ reaches `P_g` indirectly, through the adjoint chain: it enters the swing and
angle equalities, whose duals propagate back into the pre-fault coupling `Pe_init`
(ν) and the first swing step. Those two are the `dynamic` column — and if the
corridor never binds, that column is zero, which is the honest answer rather than a
missing term. Check `max|dual|` on the corridor CSVs before reading any stability
rent into λ.

What this notebook gives is therefore the exact split of λ into **cost, capacity
rent and transient coupling**. Attributing the dynamic part back to individual
corridor duals requires the implicit-function-theorem sensitivity `S = ∂h/∂P` and
the `F_x` assembly — deliberately out of scope here.